In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'genxii-ad-starting-feats-2'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

boto3==1.24.59
pandas==1.2.4
scikit_learn==0.24.1
tqdm==4.64.1

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd
import numpy as np
import json
import boto3
import time
from tqdm import tqdm

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_model = '01_ad'
    
    # get df_hyperparameters
    str_filename = 'df_hyperparameters.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/12_step_function/{str_filename}'
    df = pd.read_csv(str_uri)
    # convert to dict
    dict_hyperparameters = dict(zip(df['keys'], df['values']))
    
    # get eval metric
    str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
    print(f'Eval metric: {str_eval_metric}')
        
    # load output from shared feature selection
    print('Loading output from shared feature selection...')
    str_filename = 'df_iterative_feat_select.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/01_feat_select/04_batch_feature_selection/results/{str_filename}'
    df_feat_select = pd.read_csv(str_uri)
    
    # get max n feats and remove it
    print('Removing max n feats...')
    int_max_feats = np.max(df_feat_select['n_feats'])
    df_feat_select = df_feat_select[df_feat_select['n_feats'] < int_max_feats]
    
    # get max n feats and remove it (doing it again for this one time)
    print('Removing max n feats...')
    int_max_feats = np.max(df_feat_select['n_feats'])
    df_feat_select = df_feat_select[df_feat_select['n_feats'] < int_max_feats]
    
    # logic for sorting
    print('Sorting...')
    if str_eval_metric in ['AUC','PRAUC','F1']:
        bool_ascending = False
    else:
        bool_ascending = True
    df_feat_select.sort_values(by='flt_eval_metric_valid', ascending=bool_ascending, inplace=True)

    # get list of best features
    list_cols_model = eval(df_feat_select['list_cols_model'].iloc[0])
    # rm vehicle type
    list_cols_model = [col for col in list_cols_model if col != 'strvehicletype__app']
    # rvlr
    list_cols_model = [col for col in list_cols_model if col != 'rvlr15__tu']
    list_cols_model = [col for col in list_cols_model if col != 'rvlr17__tu']
    list_cols_model = [col for col in list_cols_model if col != 'rvlr19__tu']
    # previously dropped
    list_cols_model = [col for col in list_cols_model if col != 'linka047__tu']
    list_cols_model = [col for col in list_cols_model if col != 'derogcount__ln']
    
    # load in the list of features to drop
    print('Loading list of features to drop...')
    str_filename = 'df_feats_to_drop.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    try:
        list_feats_drop = list(pd.read_csv(str_uri)['feature'])
    except:
        # create df_feats_to_drop
        list_feats_drop = []
        df_feats_to_drop = pd.DataFrame({'feature': list_feats_drop})
        # write
        df_feats_to_drop.to_csv(str_uri, index=False)
    
    # remove the list of features to drop
    print('Removing the list of features to drop...')
    list_cols_model = [col for col in list_cols_model if col not in list_feats_drop]
    
    # write to s3
    print('Writing list of columns in model to s3...')
    df_cols_in_model = pd.DataFrame({'feature': list_cols_model})
    str_filename = 'df_cols_in_model.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    df_cols_in_model.to_csv(str_uri, index=False)
    
    # write to s3 as json for map in step function
    print('Writing json of columns in model to s3...')
    str_list_cols_model = json.dumps(list_cols_model)
    cls_client_s3 = boto3.client('s3')
    str_filename = 'json_cols_in_model.json'
    str_key = f'{str_model}/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    cls_client_s3.put_object(
        Bucket=str_project,
        Key=str_key,
        Body=str_list_cols_model,
    )

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-ad-starting-feats-2

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  45.06kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 3cd81ffec4d9
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> d0a1723b9286
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> 9c6429727067
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> dc78f875daed
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> 1e5a10ed7e96
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 04bbd969608e
Removing intermediate container 04bbd969608e
 ---> e08225989c6f
Successfully built e08225989c6f
Successfully tagged genxii-ad-starting-feats-2:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-ad-starting-feats-2' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-starting-feats-2]
b57dd24cae08: Preparing
ec59735d5aa6: Preparing
5dc878b31959: Preparing
99006831cc42: Preparing
e92756f7b561: Preparing
4fe51bf0bf5c: Preparing
fbbd8c1e2ec1: Preparing
fe2359fe88f2: Preparing
e703f2e518cc: Preparing
4fe51bf0bf5c: Waiting
97a787951169: Preparing
fbbd8c1e2ec1: Waiting
fe2359fe88f2: Waiting
e703f2e518cc: Waiting
97a787951169: Waiting
ec59735d5aa6: Layer already exists
e92756f7b561: Layer already exists
99006831cc42: Layer already exists
5dc878b31959: Layer already exists
4fe51bf0bf5c: Layer already exists
fe2359fe88f2: Layer already exists
fbbd8c1e2ec1: Layer already exists
e703f2e518cc: Layer already exists
97a787951169: Layer already exists
b57dd24cae08: Pushed
latest: digest: sha256:5e02521c9a2b507ecebc85938de8c9bcb47b1fb2319460dfe91cf71736d476db size: 2420


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 06 May 2024 16:58:04 GMT',
                                      'x-amzn-requestid': '6929819b-3a1c-4a40-966f-3390d52e18d9'},
                      'HTTPStatusCode': 204,
                      'RequestId': '6929819b-3a1c-4a40-966f-3390d52e18d9',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = '836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-starting-feats-2:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '5e02521c9a2b507ecebc85938de8c9bcb47b1fb2319460dfe91cf71736d476db',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-starting-feats-2',
 'FunctionName': 'genxii-ad-starting-feats-2',
 'LastModified': '2024-05-06T16:58:04.430+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-ad-starting-feats-2'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1211',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 06 May 2024 16:58:05 GMT',
                                      'x-amzn-requestid': '87da4f31-4458-4b13-852c-4c4d9c0e92cf'},
                      'HTTPStatusCode': 201,
                      'RequestId': '87da

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)